# J.A.R.V.I.S. — CLONALG Parameter Study

This notebook imports the existing `src/clonalg.py` without modifying it. It trains classes incrementally in this order: MNIST 0–9, then Fashion-MNIST 10–19. It evaluates after every class and logs one CSV row per stage.

**Study defaults:** 6,000 training images/class, 1,000 test images/class, 15 epochs/class, seeds `[7, 21, 42]`. The study uses a 19-configuration screening plan (57 full runs; 1,140 data rows plus the CSV header).

**Important computational note:** `batch_size` is a real algorithm parameter. Values 16, 32, and 64 mean many batches per class and may be substantially slower than `None` (one batch per class). Benchmark the baseline before starting all runs.


## 1. Imports and project paths

In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import csv, json, sys
import numpy as np
from sklearn.datasets import fetch_openml

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "experiments":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src" / "clonalg.py").exists() and (PROJECT_ROOT / "project" / "src" / "clonalg.py").exists():
    PROJECT_ROOT = PROJECT_ROOT / "project"
if not (PROJECT_ROOT / "src" / "clonalg.py").exists():
    raise FileNotFoundError("Open this notebook from the project's experiments folder or update PROJECT_ROOT.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.clonalg import CLONALG

EXPERIMENT_DIR = PROJECT_ROOT / "experiments"
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = EXPERIMENT_DIR / "clonalg_runs.csv"
SUMMARY_PATH = EXPERIMENT_DIR / "clonalg_summary.txt"
print("Project root:", PROJECT_ROOT)


## 2. Study settings and editable parameter values\nThe search is intentionally a **screening design**, not a full Cartesian product. Edit the values in this cell; the configuration builder below keeps the run count manageable and requires no changes elsewhere.

In [ ]:
TRAIN_SAMPLES_PER_CLASS = 6000
TEST_SAMPLES_PER_CLASS = 1000
EPOCHS = 15
CLASS_ORDER = list(range(20))
SEEDS = [7, 42]

# Baseline configuration
BASELINE = {
    "population_size": 20,
    "memory_size": 10,
    "selection_size": 3,
    "num_clones": 3,
    "mutation_rate": 0.2,
    "similarity_threshold": 7.5,
    "n_neighbors": 3,
    "weights": "distance",
    "metric": "euclidean",
    "batch_size": 32,
    "alpha_max": 0.5,
    "noise_std": 0.1,
}

# One-at-a-time screening values: low/high for six key parameters.
# Baseline itself is automatically included.
SCREENING_VALUES = {
    "population_size": [10, 30],
    "memory_size": [5, 15],
    "mutation_rate": [0.1, 0.5],
    "similarity_threshold": [6.0, 11.0],
    "batch_size": [16, 64],
    "noise_std": [0.05, 0.3],
    # One additional screening value for each of the remaining parameters:
    "selection_size": [1],
    "num_clones": [5],
    "n_neighbors": [7],
    "alpha_max": [0.2],
}

# Two interaction configurations, selected to probe plausible parameter interactions.
# Each is a complete parameter dictionary; unspecified values inherit BASELINE.
INTERACTION_OVERRIDES = [
    {"population_size": 30, "memory_size": 15},
    {"mutation_rate": 0.1, "noise_std": 0.3},
]


In [ ]:
def build_configurations(baseline, screening_values, interaction_overrides):
    configs = [("baseline", dict(baseline))]
    for parameter, values in screening_values.items():
        for value in values:
            config = dict(baseline)
            config[parameter] = value
            configs.append((f"{parameter}={value}", config))
    for i, overrides in enumerate(interaction_overrides, start=1):
        config = dict(baseline)
        config.update(overrides)
        configs.append((f"interaction_{i}", config))

    # Guard against accidentally duplicated configurations.
    unique = []
    seen = set()
    for label, config in configs:
        key = json.dumps(config, sort_keys=True)
        if key not in seen:
            seen.add(key)
            unique.append((label, config))
    return unique

CONFIGURATIONS = build_configurations(BASELINE, SCREENING_VALUES, INTERACTION_OVERRIDES)
TOTAL_RUNS = len(CONFIGURATIONS) * len(SEEDS)
print(f"Configurations: {len(CONFIGURATIONS)}")
print(f"Seeds/configuration: {len(SEEDS)}")
print(f"Full runs: {TOTAL_RUNS}")
print(f"Expected data rows: {TOTAL_RUNS * len(CLASS_ORDER)}")
for i, (label, params) in enumerate(CONFIGURATIONS, 1):
    print(f"{i:02d}. {label}: {params}")
assert len(CONFIGURATIONS) == 19, f"Expected 19 unique configurations, got {len(CONFIGURATIONS)}"


## 3. Load MNIST and Fashion-MNIST\nThis cell uses the project's cache if it contains the expected arrays. Otherwise, it fetches both datasets from OpenML. The label convention is MNIST 0–9 and Fashion-MNIST 10–19.

In [ ]:
CACHE_DIR = PROJECT_ROOT / "data" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_FILE = CACHE_DIR / "datasets.npz"

def load_datasets():
    if CACHE_FILE.exists():
        with np.load(CACHE_FILE) as archive:
            keys = set(archive.files)
            if {"x_train", "y_train", "x_test", "y_test"}.issubset(keys):
                print("Using dataset cache:", CACHE_FILE)
                return {k: archive[k] for k in ("x_train", "y_train", "x_test", "y_test")}

    print("Fetching MNIST and Fashion-MNIST from OpenML; first download may take a while.")
    x_m, y_m = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True,
                            data_home=str(CACHE_DIR / "openml"))
    x_f, y_f = fetch_openml("Fashion-MNIST", version=1, as_frame=False, return_X_y=True,
                            data_home=str(CACHE_DIR / "openml"))
    x_m, y_m = x_m.astype(np.uint8).reshape(-1, 28, 28), y_m.astype(np.int64)
    x_f, y_f = x_f.astype(np.uint8).reshape(-1, 28, 28), y_f.astype(np.int64)
    result = {
        "x_train": np.concatenate([x_m[:60000], x_f[:60000]]),
        "y_train": np.concatenate([y_m[:60000], y_f[:60000] + 10]),
        "x_test": np.concatenate([x_m[60000:], x_f[60000:]]),
        "y_test": np.concatenate([y_m[60000:], y_f[60000:] + 10]),
    }
    np.savez_compressed(CACHE_FILE, **result)
    return result

data = load_datasets()
X_TRAIN_RAW, Y_TRAIN_ALL = data["x_train"], data["y_train"].astype(np.int64)
X_TEST_RAW, Y_TEST_ALL = data["x_test"], data["y_test"].astype(np.int64)
print("Train:", X_TRAIN_RAW.shape, "Test:", X_TEST_RAW.shape)


## 4. Prepare per-class data and CSV logging

In [ ]:
def choose_indices(labels, cls, count, seed):
    indices = np.flatnonzero(labels == cls)
    if count is None or count >= len(indices):
        return indices
    rng = np.random.default_rng(seed + int(cls) * 1009)
    return np.sort(rng.choice(indices, size=count, replace=False))

TRAIN_BY_CLASS, TEST_BY_CLASS = {}, {}
for cls in CLASS_ORDER:
    ti = choose_indices(Y_TRAIN_ALL, cls, TRAIN_SAMPLES_PER_CLASS, 100)
    vi = choose_indices(Y_TEST_ALL, cls, TEST_SAMPLES_PER_CLASS, 200)
    TRAIN_BY_CLASS[cls] = (X_TRAIN_RAW[ti].astype(np.float32) / 255.0, Y_TRAIN_ALL[ti])
    TEST_BY_CLASS[cls] = (X_TEST_RAW[vi].astype(np.float32) / 255.0, Y_TEST_ALL[vi])

CSV_FIELDS = [
    "timestamp_utc", "config_id", "config_label", "seed", "stage", "classes_seen",
    "scope", "accuracy", "balanced_accuracy", "per_class_accuracy_json",
    "parameters_json", "elapsed_seconds"
]

def initialise_csv():
    with CSV_PATH.open("w", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=CSV_FIELDS).writeheader()

def append_stage_row(row):
    with CSV_PATH.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS)
        writer.writerow(row)
        f.flush()

print("Samples per class (train/test):",
      len(TRAIN_BY_CLASS[0][1]), "/", len(TEST_BY_CLASS[0][1]))


## 5. Run experiment\nEach configuration/seed starts with a fresh CLONALG instance. After every class, predictions are evaluated over all classes seen so far. Predictions are chunked to limit memory use. Each completed stage is written immediately to the CSV.

In [ ]:
def evaluate_seen_classes(model, seen_classes):
    x_eval = np.concatenate([TEST_BY_CLASS[c][0] for c in seen_classes])
    y_true = np.concatenate([TEST_BY_CLASS[c][1] for c in seen_classes])
    y_pred = np.concatenate([
        model.predict(x_eval[start:start + 512])
        for start in range(0, len(x_eval), 512)
    ])
    per_class = {}
    for cls in seen_classes:
        mask = y_true == cls
        per_class[str(cls)] = float(np.mean(y_pred[mask] == y_true[mask]))
    return float(np.mean(y_pred == y_true)), float(np.mean(list(per_class.values()))), per_class

def read_rows():
    if not CSV_PATH.exists():
        return []
    with CSV_PATH.open("r", newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def write_summary():
    rows = read_rows()
    def rank_at_stage(stage):
        grouped = {}
        for row in rows:
            if int(row["stage"]) == stage:
                grouped.setdefault(row["config_id"], []).append(row)
        ranking = []
        for config_id, recs in grouped.items():
            # Only rank a config if all requested seeds reached this stage.
            seed_scores = {int(r["seed"]): float(r["accuracy"]) for r in recs}
            if not set(SEEDS).issubset(seed_scores):
                continue
            scores = [seed_scores[s] for s in SEEDS]
            ranking.append({
                "config_id": config_id, "label": recs[0]["config_label"],
                "mean": float(np.mean(scores)), "std": float(np.std(scores, ddof=1)),
                "scores": seed_scores, "params": json.loads(recs[0]["parameters_json"])
            })
        return sorted(ranking, key=lambda x: x["mean"], reverse=True)

    mnist = rank_at_stage(10)
    overall = rank_at_stage(20)
    lines = [
        "J.A.R.V.I.S. CLONALG — Parameter Study Summary",
        f"Generated: {datetime.now(timezone.utc).isoformat()}",
        f"Training samples/class: {TRAIN_SAMPLES_PER_CLASS}",
        f"Test samples/class: {TEST_SAMPLES_PER_CLASS}",
        f"Epochs/class: {EPOCHS}",
        f"Seeds: {SEEDS}",
        f"Configurations: {len(CONFIGURATIONS)}",
        "",
    ]
    for title, ranking in [("BEST AFTER MNIST (10 CLASSES)", mnist),
                           ("BEST AFTER ALL 20 CLASSES", overall)]:
        lines.append(title)
        if not ranking:
            lines.append("No configuration has completed all seeds for this stage.")
        else:
            best = ranking[0]
            lines.extend([
                f"Configuration: {best['config_id']} ({best['label']})",
                f"Mean accuracy: {best['mean']:.6f}",
                f"Standard deviation: {best['std']:.6f}",
                f"Per-seed accuracy: {json.dumps(best['scores'], sort_keys=True)}",
                f"Parameters: {json.dumps(best['params'], sort_keys=True)}",
            ])
        lines.append("")
        lines.append("RANKING")
        for rank, item in enumerate(ranking, 1):
            lines.append(f"{rank}. {item['config_id']} ({item['label']}): mean={item['mean']:.6f}, std={item['std']:.6f}")
        lines.append("")
    SUMMARY_PATH.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("Summary written:", SUMMARY_PATH)

def run_study():
    initialise_csv()  # Fresh CSV for this complete study.
    started = perf_counter()
    for config_number, (label, params) in enumerate(CONFIGURATIONS, 1):
        config_id = f"config_{config_number:02d}"
        for seed in SEEDS:
            print(f"\n{config_id} — {label} — seed {seed}")
            np.random.seed(int(seed))
            model = CLONALG(**params, callback=None)
            run_started = perf_counter()
            seen = []
            for cls in CLASS_ORDER:
                x_train, y_train = TRAIN_BY_CLASS[cls]
                model.partial_fit(x_train, y_train, epochs=EPOCHS, eval_data=None, class_order=[cls])
                seen.append(cls)
                accuracy, balanced, per_class = evaluate_seen_classes(model, seen)
                stage = len(seen)
                scope = "MNIST_only" if stage == 10 else ("all_20_classes" if stage == 20 else "incremental")
                append_stage_row({
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                    "config_id": config_id, "config_label": label, "seed": seed,
                    "stage": stage, "classes_seen": json.dumps(seen), "scope": scope,
                    "accuracy": f"{accuracy:.8f}", "balanced_accuracy": f"{balanced:.8f}",
                    "per_class_accuracy_json": json.dumps(per_class, sort_keys=True),
                    "parameters_json": json.dumps(params, sort_keys=True),
                    "elapsed_seconds": f"{perf_counter() - run_started:.3f}",
                })
                print(f"  stage {stage:02d}/20 accuracy={accuracy:.4f}, balanced={balanced:.4f}")
            write_summary()  # Keep summary up to date after every complete run.
    print(f"Study finished in {(perf_counter() - started) / 60:.1f} minutes.")
    write_summary()
    return read_rows()

# Launch the full study
rows = run_study()


## 6. Resume / inspect\nThe full run cell starts a fresh CSV. If the kernel stops, do not rerun it immediately unless you want to overwrite the log. You can inspect existing completed rows below and call `write_summary()` to rebuild the summary from the current CSV.

In [ ]:
# Optional: inspect existing progress without starting training.
# rows = read_rows()
# print("Logged stage rows:", len(rows))
# write_summary()
